# 12. Successor linkage and current development-reference evaluation

Generated: `2026-08-14T11:16:53`

This notebook is regenerated from the current script outputs. It is the reader-facing linkage/evaluation notebook for the current evidence state.

## tl;dr

`M_B_text_ranking @ 0.70` is the frozen conservative primary event definition, not a claim of threshold optimality. The latest internal held-out comparison includes all four algorithms, including `M_D_fellegi_sunter`, which is now scored from the fitted model.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'scripts').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
PROCESSED = PROJECT_ROOT / 'data/processed/boamp'
BENCHMARK = PROCESSED / 'benchmark'

def load_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

dev = load_json(PROCESSED / 'linkage_evaluation_dev.json')
validation = load_json(PROCESSED / 'linkage_evaluation_validation.json')
modeling = load_json(BENCHMARK / 'modeling/modeling_summary.json')
manifest = load_json(BENCHMARK / 'benchmark_manifest.json')


In [ ]:
def method_frame(summary):
    rows = []
    for method in summary['methods']:
        metrics = method['unweighted_all_frames']
        weighted = method.get('weighted_national', {})
        rows.append({
            'method': method['method'],
            'threshold': method['threshold'],
            'accepted_links': metrics['accepted_links'],
            'precision': metrics['precision_at_1'],
            'recall': metrics['recall_at_1'],
            'fpr': metrics['false_positive_rate_on_negatives'],
            'coverage': metrics['coverage'],
            'weighted_precision': weighted.get('precision_at_1', {}).get('estimate'),
            'weighted_recall': weighted.get('recall_at_1', {}).get('estimate'),
            'weighted_fpr': weighted.get('false_positive_rate_on_verified_negatives', {}).get('estimate'),
        })
    return pd.DataFrame(rows)

dev_methods = method_frame(dev)
validation_methods = method_frame(validation)
validation_methods


## Benchmark State

In [ ]:
pd.DataFrame([
    {'item': 'labelled anchors', 'value': manifest['anchor_totals']['anchors']},
    {'item': 'labelled pairs', 'value': manifest['anchor_totals']['labelled_pairs']},
    {'item': 'dev anchors', 'value': modeling['outputs']['dev']['anchors']},
    {'item': 'dev rows', 'value': modeling['outputs']['dev']['rows']},
    {'item': 'validation anchors', 'value': modeling['outputs']['validation']['anchors']},
    {'item': 'validation rows', 'value': modeling['outputs']['validation']['rows']},
])

## Internal Held-Out Method Comparison

In [ ]:
display(validation_methods[['method', 'threshold', 'accepted_links', 'precision', 'recall', 'fpr', 'coverage']])

ax = validation_methods.set_index('method')[['precision', 'recall', 'fpr']].plot(
    kind='bar', figsize=(9, 4.5), width=0.72
)
ax.set_title('Current internal reference metrics')
ax.set_ylabel('rate')
ax.set_ylim(0, 1)
ax.set_xlabel('')
ax.legend(['precision@1', 'recall@1', 'FPR on negatives'], frameon=False)
ax.tick_params(axis='x', rotation=28)
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()

## Interpretation

`M_C_weighted_gated` recovers more true successors, but its false-positive rate is materially higher. For survival analysis, a false link is more damaging than an abstention because it fabricates both an event and an event time. Threshold `0.60` performs better on the small bootstrap validation split but worse on development precision and FPR, so it remains a sensitivity arm rather than being promoted post hoc. The use of precision-recall evidence for this rare-positive decision follows [Davis and Goadrich (2006)](https://doi.org/10.1145/1143844.1143874) and [Saito and Rehmsmeier (2015)](https://doi.org/10.1371/journal.pone.0118432). Those papers support the diagnostic choice, not this project's numerical results.

## Modeling-Ready Tables

In [ ]:
pd.DataFrame(modeling['outputs']).T[[
    'rows', 'anchors', 'probability_frame_anchors', 'primary_positive_pairs',
    'strict_positive_pairs', 'broad_positive_pairs', 'verified_negative_anchors'
]]

In [ ]:
feature_columns = pd.Series(modeling['feature_columns'], name='feature')
display(feature_columns.to_frame())
assert 'fs_match_probability' in set(modeling['feature_columns'])


## Caveat

The current labels were generated by deterministic bootstrap rules. The two passes are not independent annotations, so their agreement is self-consistency rather than specialist inter-annotator agreement. These metrics are development evidence, not validated legal-renewal accuracy.